In [2]:
from math import exp, log, sin, cos
import random

class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None # Returns none because gradients should start at 0 when computing derivatives
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other): 
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  # Multiplication
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self,other), '*')

    def _backward():
      other.grad += self.data * out.grad
      self.grad += other.data * out.grad
    out._backward = _backward

    return out
  
  #Euler
  def exp(self):
    x = max(-700, min(700, self.data)) 
    out = Value(exp(x), (self, ), 'exp')
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out
  
  #Logarithm
  def log(self):
    out = Value(log(self.data), (self, ), "log")
    
    def _backward():
      self.grad += (1/self.data) * out.grad
      
    out._backward = _backward
    return out
  
  # Power
  def __pow__(self, other):
    assert isinstance(other, (int, float))
    out = Value(self.data**other, (self, ), "^")

    def _backward():
      self.grad += (other*(self.data**(other-1))) * out.grad
    out._backward = _backward
    return out
  
  # Sin
  def sin(self):
    out = Value(sin(self.data), (self, ), "sin")
  
    def _backward():
      self.grad += (cos(self.data)) * out.grad
    
    out._backward = _backward
    return out
  
  #Tanh
  def tanh(self):
    e2x = (self * 2).exp()
    return (e2x - 1) / (e2x + 1)
  
  def __sub__(self, other):
    return self + (-other)
  def __radd__(self, other):
    return self + other
  def __rmul__(self, other):
    return self * other
  def __rsub__(self, other):
    return self - other
  def __neg__(self):
    return self * -1
  def __truediv__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    return self *(other**-1)
  def __rtruediv__(self, other): 
    return Value(other) * (self**-1)

  def backward(self): 
    topo = [] #Creates an empty list
    visited = set() #Prevents nodes from repeating
    def build_topo(v):
      if v not in visited:
        visited.add(v) #Append to the set so that it does not repeat
        for child in v._prev: #Takes the values that amount to output v
          build_topo(child) #Repeat until every child is accounted for
        topo.append(v) #Append the first child before the parent
    build_topo(self) 

    self.grad = 1.0 #Derivative of final output
    for node in reversed(topo): #Reversed because we want to start from the top
      node._backward() #Calls the backward pass which is the function to get the gradient of the out value

In [3]:
class Module:
    def zero_grad(self, params):
        for p in params:
            p.grad = 0.0
    
    def parameters(self):
        return []

class Neuron(Module):
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
        
    def __call__(self, x):
        act = sum( wi*xi for wi, xi in zip(self.w, x)) + self.b
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]
    
class Layer(Module):
    def __init__(self,nin,nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
        
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
    
class MLP(Module):
    def __init__(self,nin,nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [4]:
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=50, noise=0.2, random_state=42)
y

array([0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0,
       0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1,
       0, 0, 1, 0, 1, 1])

In [6]:
n = MLP(50,[50,25,10,1])

for k in range(20):
    # forward pass
    ypred = [n(x) for x in X]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(y, ypred))

    # backward pass
    n.zero_grad(n.parameters())
    loss.backward()

    # update
    for p in n.parameters():
        p.data += -0.002 * p.grad

    print(k, loss.data)

0 39.48237700742545
1 13.825674089450475
2 9.755589917298956
3 8.865899448887705
4 8.341259056678874
5 7.963139967876718
6 7.62801408534954
7 7.406228037041691
8 7.161741272251529
9 7.01992883697282
10 6.805764973074596
11 6.684865434171087
12 6.483708878913746
13 6.363829868470434
14 6.180618536845994
15 6.061883678261116
16 5.9000490224838185
17 5.7854767137000715
18 5.642697111146561
19 5.533044475380415
